## URI Analysis Skript und Playground für die Distributionsanalysse

In diesem notebook werden einige Ideen getestet, um eine möglichst Aussagekräftige Analyse von URI-Fetch-Results zu ermöglichen.

Ziel ist es, zu Beginn erstmal grob die returnierten HTTP-Statuscodes zu analysieren, um zu schauen ob der Link überhaupt erreichbar ist.

Desweiteren wäre es von Nutzen eine Methode zu implementieren, die es ermöglicht folgende Dinge aus dem Result zu bekommen:

- Ist das Result ein direkter Dateidownload
- Wenn ja wie groß ist die Datei
- Von welchem Typ ist die Datei
- Einen kleinen Ausschnitt der Datei zwischenspeichern

- Wenn nein. Was ist in der Response? HTML, XML eines WMF Endpunkts, REST



Zum anderen soll in diesem Notebook geprüft werden wie man generell am besten die Distributionen überprüft

Was soll gemacht werden wenn es abartig viele Distributionen sind? Teilweise sind es über 50 50 bis sogar 100 Distributionen. wie soll sowas überprüft werden.

Wie gibt man bei einem solchen Fall Kontext für ein LLM-Agent mit?

Wie soll zwischen Geo und Non Geo Datensätzen unterschieden werden?
- Im Interview I2 haben wir erfahren dass die Distributionen eigentlich lediglich unterschiedliche "Distributionen" des gleichen Datensatzes sein sollten.
- Das Problem an der Sache ist bei Geodaten sind die Hauptdistributionen fast immer WMS WFS Endpunkte also mehr oder weniger nochmal ein Verweis auf Metadaten über die man dann auf die eigentlichen Daten kommt.
- Heisst Geodaten sind dann in diesem Punkt immer falsch? 
- Wie weit sollte man den ganzen Punkt Distributionen überhaupt mit in die Analyse packen?

In [12]:
# Wir fangen an mit ein paar einfachen URI-fetches und schauen uns mal an wie das so zurück kommt.


URI_CSV_DOWNLOAD = "https://offenedaten.aachen.de/dataset/6cbe0cbe-b1d6-4755-9e76-b1cec04bd857/resource/24b698d1-a72d-464d-8887-5cafec653ca8/download/2021_november.csv"

URI_GEO_JSON_DOWNLOAD = "https://geoserver.digitale-mrn.de/geoserver/metropolatlas/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=metropolatlas%3AMRN_LDI_PREASSURE_4326_Point&outputFormat=application%2Fjson"

URI_WMS_GET_CAPABILITIES = "https://geoserver.digitale-mrn.de/geoserver/metropolatlas/MRN_LDI_PREASSURE_4326_Point/wms?request=GetCapabilities"

URI_GPKG_DOWNLOAD = "https://geoserver.digitale-mrn.de/geoserver/metropolatlas/ows?service=WFS&version=1.0.0&request=GetFeature&typeName=metropolatlas%3AMRN_LDI_PREASSURE_4326_Point&outputFormat=application%2Fx-gpkg"

URI_PDF_DOWNLOAD = "https://www.offenbach.de/medien/bindata/of/Statistik_und_wahlen_/dir-18/STAK-2024-Statistik-Aktuell-Nr.24-Zensus-2022-in-Offenbach-erste-Ergebnisse.pdf"

URI_XLSX_DOWNLOAD = "https://www.offenbach.de/medien/bindata/of/Statistik_und_wahlen_/dir-18/dir-38/20240625_Zensus-2022-Ergebnis-f.Offenbach-am-Main-Grundinfo-Bevoelkerung.xlsx"

URI_WFS_GET_CAPABILITIES = "https://geoportal.freiburg.de/wfs/gut_parken/gut_parken?REQUEST=GetCapabilities&SERVICE=WFS"

URI_500_ERROR_RESPONSE = "https://portal-chemnitz.opendata.arcgis.com/datasets/chemnitz::spielplaetzepunkte.zip?outSR=%7B%22latestWkid%22%253A3857%252C%22wkid%22%253A102100%7D"

URI_400_ERROR_RESPONSE = "https://hub.arcgis.com/api/download/v1/items/4c331993dab54b49bbc9debfc5928ec3/zip?redirect=true&layers=0"

URI_CSV_DOWNLOAD_2 = "https://www.statistikdaten.bayern.de/genesis/online?operation=download&code=12612-003z&option=csv&regionalschluessel=09377117&regionalmerkmal=GEMEIN"




def fetch_uri(uri):

    import requests

    try:
        response = requests.get(uri)
        print(f"URI: {uri}")
        print(f"Status Code: {response.status_code}")
        print(f"Content-Type: {response.headers.get('Content-Type')}")
        print("-" * 40)
    except Exception as e:
        print(f"Error fetching URI: {uri}")
        print(f"Error: {e}")
        print("-" * 40)



uris = [
    URI_CSV_DOWNLOAD,
    URI_GEO_JSON_DOWNLOAD,
    URI_WMS_GET_CAPABILITIES,
    URI_GPKG_DOWNLOAD,
    URI_PDF_DOWNLOAD,
    URI_XLSX_DOWNLOAD,
    URI_WFS_GET_CAPABILITIES,
    URI_500_ERROR_RESPONSE,
    URI_400_ERROR_RESPONSE
]


fetch_uri(URI_CSV_DOWNLOAD_2)

URI: https://www.statistikdaten.bayern.de/genesis/online?operation=download&code=12612-003z&option=csv&regionalschluessel=09377117&regionalmerkmal=GEMEIN
Status Code: 200
Content-Type: text/html;charset=UTF-8
----------------------------------------


Als nächstes probieren wir eine Lösung von github für die Ressourcetyp validierung für eine rdf File von govdata.de umzubauen.

In [16]:
import mimetypes
import os
import re
from urllib.parse import urlparse
import requests


def inspect_download_url(url, timeout=30):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*",
    }

    with requests.Session() as session:
        response = session.get(
            url,
            headers=headers,
            stream=True,
            allow_redirects=True,
            timeout=timeout,
        )

        content_type = response.headers.get("Content-Type", "")
        content_disposition = response.headers.get("Content-Disposition", "")
        content_length = response.headers.get("Content-Length")

        # Read a small sample without downloading the entire file
        sample = next(response.iter_content(chunk_size=2048), b"")

    filename = None

    # Try filename from Content-Disposition
    match = re.search(
        r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?',
        content_disposition,
        flags=re.IGNORECASE,
    )
    if match:
        filename = match.group(1)

    # Fallback: filename from URL path
    if not filename:
        path = urlparse(response.url).path
        basename = os.path.basename(path)
        if "." in basename:
            filename = basename

    extension = None
    if filename:
        extension = os.path.splitext(filename)[1].lower()

    # Guess from Content-Type
    guessed_extension = mimetypes.guess_extension(content_type.split(";")[0].strip())

    # Simple body sniffing
    sample_lower = sample[:500].lower().lstrip()

    looks_like_html = (
        sample_lower.startswith(b"<!doctype html")
        or sample_lower.startswith(b"<html")
        or b"<head" in sample_lower
        or b"<body" in sample_lower
    )

    looks_like_csv = False
    try:
        decoded = sample.decode("utf-8", errors="ignore")
        first_line = decoded.splitlines()[0] if decoded.splitlines() else ""
        looks_like_csv = ";" in first_line or "," in first_line or "\t" in first_line
    except Exception:
        pass

    is_download = bool(content_disposition) or (
        not looks_like_html
        and content_type
        and "text/html" not in content_type.lower()
    )

    return {
        "status_code": response.status_code,
        "final_url": response.url,
        "content_type": content_type,
        "content_disposition": content_disposition,
        "content_length": content_length,
        "filename": filename,
        "extension_from_filename": extension,
        "extension_from_content_type": guessed_extension,
        "looks_like_html": looks_like_html,
        "looks_like_csv": looks_like_csv,
        "probably_direct_download": is_download,
        "sample_text": sample[:500].decode("utf-8", errors="replace"),
    }



url = "https://geodienste.hamburg.de/HH_WFS_xplan_dls?service=WFS&request=GetFeature&version=2.0.0&resolvedepth=%252A&StoredQuery_ID=urn:ogc:def:query:OGC-WFS::PlanName&planName=TB40"

info = inspect_download_url(url)

for key, value in info.items():
    print(f"{key}: {value}")

status_code: 400
final_url: https://geodienste.hamburg.de/HH_WFS_xplan_dls?service=WFS&request=GetFeature&version=2.0.0&resolvedepth=%252A&StoredQuery_ID=urn:ogc:def:query:OGC-WFS::PlanName&planName=TB40
content_type: application/xml;charset=UTF-8
content_disposition: 
content_length: None
filename: None
extension_from_filename: None
extension_from_content_type: .xsl
looks_like_html: False
looks_like_csv: False
probably_direct_download: True
sample_text: <?xml version='1.0' encoding='UTF-8'?>
<ows:ExceptionReport xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ows/1.1 http://schemas.opengis.net/ows/1.1.0/owsExceptionReport.xsd" version="2.0.0">
  <ows:Exception exceptionCode="InvalidParameterValue">
    <ows:ExceptionText>Value of parameter traverseXlinkDepth (='%2A') is neither a positive integer nor '*'.</ows:ExceptionText>
  </ows:Exception>
</ows:Exce
